# VocaBloom tutor: ince ayar (fine-tuning)

Bu not defteri VocaBloom'un üç çevrimdışı yapay zekâ paketini (Hafif, Gelişmiş, Profesyonel) İngilizce-Türkçe öğretmenlik verisiyle eğitir ve telefonda çalışan GGUF dosyalarını üretir.

**Ücretsiz çalıştırma:** Kaggle (haftada 30 saat ücretsiz GPU; *Settings → Accelerator → GPU T4 x2* ve *Internet on*) veya Google Colab (ücretsiz T4). Ücretli bir şey gerekmez.

Süre (T4, ~3.000 örnek, 2 tur): Hafif ~15 dk, Gelişmiş ~40 dk, Profesyonel ~2 saat.

Hugging Face'e yüklemek için Kaggle'da *Add-ons → Secrets* altına `HF_TOKEN` (yazma izinli) ekleyin. Yüklenen dosyanın adresi uygulamada `lib/ai/packages.dart` içine yazılır.

In [ ]:
# Which package to train: 'lite' (Qwen3 0.6B), 'plus' (Qwen3 1.7B) or 'pro' (Qwen3 4B Instruct 2507)
PACKAGE = 'pro'
HF_REPO = ''   # e.g. 'kullaniciadi/vocabloom-tutor' ; empty = do not upload
EPOCHS = 2

In [ ]:
%%capture
!pip install -q unsloth
!git clone -q --depth 1 -b tutor-data https://github.com/hamitgk-beep/vocabloom || git clone -q --depth 1 https://github.com/hamitgk-beep/vocabloom

In [ ]:
import json, torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

BASE = {'lite': 'unsloth/Qwen3-0.6B', 'plus': 'unsloth/Qwen3-1.7B', 'pro': 'unsloth/Qwen3-4B-Instruct-2507'}[PACKAGE]
QUANT = {'lite': 'q6_k', 'plus': 'q6_k', 'pro': 'q5_k_m'}[PACKAGE]
HYBRID = PACKAGE != 'pro'   # Qwen3 0.6B / 1.7B have a thinking mode the app switches off
LR = {'lite': 2e-4, 'plus': 1.5e-4, 'pro': 1e-4}[PACKAGE]
RANK = {'lite': 64, 'plus': 32, 'pro': 32}[PACKAGE]

model, tokenizer = FastLanguageModel.from_pretrained(BASE, max_seq_length=4096, load_in_4bit=(PACKAGE == 'pro'), dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=RANK, lora_alpha=RANK, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=3407)

In [ ]:
def load(path):
    rows = [json.loads(l) for l in open(path, encoding='utf-8')]
    texts = []
    for r in rows:
        msgs = [dict(m) for m in r['messages']]
        if HYBRID:
            # On the phone the prompt ends with an empty think block (thinking off): train the same way.
            for m in msgs:
                if m['role'] == 'assistant':
                    m['content'] = '<think>\n\n</think>\n\n' + m['content']
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, enable_thinking=False))
    return Dataset.from_dict({'text': texts}), rows

train_ds, _ = load('vocabloom/tools/tutor/data/train.jsonl')
eval_ds, eval_rows = load('vocabloom/tools/tutor/data/eval.jsonl')
print(len(train_ds), 'training examples')
print(train_ds[0]['text'][:1500])

In [ ]:
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field='text', max_seq_length=4096, per_device_train_batch_size=4,
        gradient_accumulation_steps=4, warmup_ratio=0.05, num_train_epochs=EPOCHS, learning_rate=LR,
        lr_scheduler_type='cosine', logging_steps=10, eval_strategy='epoch', optim='adamw_8bit',
        weight_decay=0.01, seed=3407, output_dir='out', report_to='none',
        bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported()))
# Learn only the teacher's answers, not the prompts.
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
trainer.train()

In [ ]:
# Read a few answers on examples the model has not seen.
FastLanguageModel.for_inference(model)
for r in eval_rows[:6]:
    msgs = r['messages'][:-1]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, enable_thinking=False, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=ids, max_new_tokens=400, temperature=0.3, top_p=0.8, top_k=20, do_sample=True)
    print('=' * 80, '\n', msgs[-1]['content'][:300], '\n---\n', tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

In [ ]:
# GGUF for the app (same quantisation and size as the package card).
model.save_pretrained_gguf('gguf', tokenizer, quantization_method=QUANT)
!ls -la gguf

In [ ]:
import glob, os
if HF_REPO:
    from huggingface_hub import HfApi
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    api = HfApi(token=token)
    api.create_repo(HF_REPO, exist_ok=True)
    f = [x for x in glob.glob('gguf/*.gguf') if QUANT.upper() in x.upper()][0]
    name = 'vocabloom-tutor-%s-%s.gguf' % (PACKAGE, QUANT)
    api.upload_file(path_or_fileobj=f, path_in_repo=name, repo_id=HF_REPO)
    print('Uygulamaya yazılacak adres: https://huggingface.co/%s/resolve/main/%s' % (HF_REPO, name))
    print('Boyut:', os.path.getsize(f), 'bayt')